In [ ]:
# (1) Mount drive, check for GPU, CUDA driver and pytorch versions
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
import torch
print(torch.__version__)
!mkdir /content/drive/MyDrive/sent_embed_gce3/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Tue Jan 10 19:59:24 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  A100-SXM4-40GB      Off  | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0    57W / 400W |      0MiB / 40536MiB |      0%      Default |
|  

In [2]:
# (2) Clone SimCSE repo, install requirements and download dataset

%cd /content/drive/MyDrive/sent_embed_gce3/
!git clone https://github.com/princeton-nlp/SimCSE.git

%cd SimCSE
!pip install -r requirements.txt
%cd SentEval/data/downstream/
!bash download_dataset.sh

%cd /content/drive/MyDrive/sent_embed_gce3/SimCSE/data/
!bash download_nli.sh
!bash download_wiki.sh

## <Important> Replace SimCSE/simcse/models.py, SimCSE/simcse/trainers.py, and SimCSE/train.py

/content/drive/MyDrive/sent_embed_gce3
Cloning into 'SimCSE'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 434 (delta 43), reused 35 (delta 33), pack-reused 377
Receiving objects: 100% (434/434), 41.47 MiB | 24.56 MiB/s, done.
Resolving deltas: 100% (210/210), done.
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
/content/drive/MyDrive/sent_embed_gce3/SimCSE
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
/content/drive/MyDrive/sent_embed_gce3/SimCSE/SentEval/data/downstream
--2023-01-10 20:00:12--  https://huggingface.co/datasets/princeton-nlp/datasets-for-simcse/resolve/main/senteval.tar
Resolving huggingface.co (huggingface.co)... 54.235.118.239, 3.231.67.228, 18.235.116.140, ...
Connecting to huggingface.co (huggingface.co)|54.235.

In [3]:
#(3) Evaluate pretrained model to ensure that repo was correctly cloned
%cd /content/drive/MyDrive/sent_embed_gce3/SimCSE
!python evaluation.py \
    --model_name_or_path princeton-nlp/sup-simcse-bert-base-uncased \
    --pooler cls \
    --task_set sts \
    --mode test

/content/drive/MyDrive/sent_embed_gce3/SimCSE
2023-01-10 20:01:19,775 : Starting new HTTPS connection (1): huggingface.co:443
2023-01-10 20:01:19,911 : https://huggingface.co:443 "HEAD /princeton-nlp/sup-simcse-bert-base-uncased/resolve/main/config.json HTTP/1.1" 200 0
2023-01-10 20:01:19,914 : Starting new HTTPS connection (1): huggingface.co:443
2023-01-10 20:01:20,058 : https://huggingface.co:443 "HEAD /princeton-nlp/sup-simcse-bert-base-uncased/resolve/main/pytorch_model.bin HTTP/1.1" 302 0
2023-01-10 20:01:21,991 : Starting new HTTPS connection (1): huggingface.co:443
2023-01-10 20:01:22,126 : https://huggingface.co:443 "HEAD /princeton-nlp/sup-simcse-bert-base-uncased/resolve/main/config.json HTTP/1.1" 200 0
2023-01-10 20:01:22,130 : Starting new HTTPS connection (1): huggingface.co:443
2023-01-10 20:01:22,262 : https://huggingface.co:443 "HEAD /princeton-nlp/sup-simcse-bert-base-uncased/resolve/main/vocab.txt HTTP/1.1" 200 0
2023-01-10 20:01:22,265 : Starting new HTTPS connectio

In [17]:
# (4) Contrastive pretraining from bert-base, time required w/ 1 A100 GPU: [15 minutes/epoch]
%cd /content/drive/MyDrive/sent_embed_gce3/SimCSE

!python train.py \
    --model_name_or_path bert-base-uncased \
    --train_file data/nli_for_simcse.csv \
    --output_dir result/bert-base-35-256-5 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 256 \
    --learning_rate 3e-5 \
    --max_seq_length 32 \
    --gcbs_quantile 0.999 \
    --evaluation_strategy steps \
    --metric_for_best_model stsb_spearman \
    --load_best_model_at_end \
    --eval_steps 125 \
    --pooler_type cls \
    --overwrite_output_dir \
    --temp 0.05 \
    --do_train \
    --do_eval \
    --fp16 \
    "$@"

# Hyperparameters to tune (cmd line arguments)

# (1) Models: --model_name_or_path [bert-base-uncased, bert-large-uncased, roberta-base, roberta-large]
# (2) Learning rate: --learning_rate [range: (5e-6, 1e-4)]
# (3) Batch Size: --per_device_train_batch_size [64, 128, 256, 512]
# (4) GCBS Quantile: --gcbs_quantile [0.995, 0.999, 0.9995. 0.9999]
# (5) Number of Epochs: --num_train_epochs [3, 5, 7, 10]

/content/drive/MyDrive/sent_embed_gce3/SimCSE
2023-01-10 18:08:09.576663: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
100% 1/1 [00:00<00:00, 13.81it/s]
[INFO|configuration_utils.py:445] 2023-01-10 18:08:12,825 >> loading configuration file https://huggingface.co/bert-base-uncased/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/3c61d016573b14f7f008c02c4e51a366c67ab274726fe2910691e2a761acf43e.37395cee442ab11005bcd270f3c34464dc1704b715b5d7d52b1a461abe3b9e4e
[INFO|configuration_utils.py:481] 2023-01-10 18:08:12,825 >> Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "in

In [21]:
# (5) Evaluate contrastive pretraining
!python simcse_to_huggingface.py --path result/bert-base-35-256-5
%cd /content/drive/MyDrive/sent_embed_gce3/SimCSE
!python evaluation.py \
    --model_name_or_path result/bert-base-35-256-5 \
    --pooler cls \
    --task_set sts \
    --mode test

SimCSE checkpoint -> Huggingface checkpoint for result/bert-base-35-256-5
/content/drive/MyDrive/sent_embed_gce3/SimCSE
2023-01-10 19:28:50,887 : ***** Transfer task : STS12 *****


/content/drive/MyDrive/sent_embed_gce3/SimCSE/./SentEval/senteval/sts.py:42: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  sent1 = np.array([s.split() for s in sent1])[not_empty_idx]
/content/drive/MyDrive/sent_embed_gce3/SimCSE/./SentEval/senteval/sts.py:43: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  sent2 = np.array([s.split() for s in sent2])[not_empty_idx]
2023-01-